In [ ]:
import torch
import numpy as np
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv, LinAlgError
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score
from joblib import Parallel, delayed
import os
import time
TOPICS = {
    "Baseline": None,

    # Immigration / rights / institutions
    "imm_unauth": "policy toward unauthorized immigrants",
    "birthright": "birthright citizenship for children born in the United States",
    "paid_leave": "paid parental leave policies",
    "trump_corr": "political corruption involving Donald Trump",
    "journ_access": "press and journalist access to government",

    # Guns / crime / courts
    "gun_bkg_chk": "background checks for gun purchases",
    "illeg_child": "policies regarding children of undocumented immigrants",
    "death_pen": "death penalty for serious crimes",
    "abortion": "abortion legality and access",
    "scotus_abort": "Supreme Court decisions related to abortion",
    "ar_ban": "banning assault rifles",

    # Elections / economy
    "econ_now": "evaluations of the current national economy",
    "voter_id": "voter ID requirements",
    "felon_vote": "voting rights for people with felony convictions",

    # Health / education / spending
    "vax_school": "school vaccination requirements",
    "spend_border": "federal spending on border security",
    "spend_welfare": "federal spending on welfare programs",
    "spend_poor": "federal spending to aid the poor",
    "spend_school": "federal spending on public schools",

    # Governance / social issues
    "checks_power": "checks and balances among branches of government",
    "border_wall": "building a wall on the U.S.–Mexico border",
    "trans_bath": "transgender bathroom policies",
    "police_force": "police excessive use of force",
    "assist_black": "government assistance to Black Americans",
    "job_guar": "a government-guaranteed job program",
    "govt_health": "government-provided health insurance",
    "obamacare": "the Affordable Care Act (Obamacare)",

    # Spending / unrest / climate / foreign policy
    "svc_spend": "government spending on services",
    "def_spend": "government spending on national defense",
    "urban_unrest": "urban unrest and protest-related concerns",
    "clim_imp": "the importance of climate change",
    "rus_interf": "Russian interference in U.S. elections",

    # Feeling thermometers
    "ft_trans": "transgender people",
    "ft_union": "labor unions",
    "ft_fem": "feminists",
    "ft_sci": "scientists",
    "ft_police": "police",

    # Environment / guns / military
    "env_bus": "tradeoff between environmental protection and business",
    "ghg_emiss": "regulating greenhouse gas emissions",
    "gun_imp": "importance of gun regulation",
    "mil_force": "use of U.S. military force internationally",
}


# ==========================================
# 1. CONFIGURATION
# ==========================================
# A100 40GB can handle large batches for 8B models
BATCH_SIZE = 64
MODEL_PATH = "/project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct"
NOMINATE_CSV = "/project/jevans/maxzhuyt/data/HS116_members_fullname.csv"

# Topics: Core + Bipartisan/Horseshoe Candidates

# ==========================================
# 2. MODEL LOADER & EXTRACTION (GPU)
# ==========================================
def load_model(path):
    print(f"Loading model from: {path}...")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(path, use_fast=True, local_files_only=True)
    
    # CRITICAL: Left padding allows batching without destroying the last token position
    tokenizer.padding_side = 'left' 
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    model = AutoModelForCausalLM.from_pretrained(
        path, dtype=dtype, device_map="auto", local_files_only=True, attn_implementation="eager"
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer

@torch.no_grad()
def extract_heads_batched(model, tokenizer, texts, batch_size=32):
    """
    Optimized extraction for A100.
    """
    model.eval()
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads
    D_head = model.config.hidden_size // H
    
    activations = []
    
    # Pre-allocate hook containers
    layer_outputs = [None] * L
    
    def get_hook(layer_idx):
        def hook(module, input, output):
            # Input[0] shape: [Batch, Seq, Hidden]
            # Reshape to [Batch, Seq, Heads, Head_Dim]
            # We immediately move to CPU to free VRAM for the next batch
            reshaped = input[0].detach().view(input[0].shape[0], input[0].shape[1], H, D_head)
            layer_outputs[layer_idx] = reshaped[:, -1, :, :].float().cpu().numpy()
        return hook

    # Register hooks once
    hooks = []
    for li in range(L):
        hooks.append(model.model.layers[li].self_attn.o_proj.register_forward_hook(get_hook(li)))

    print(f"  > Extracting with Batch Size {batch_size}...")
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Fast Tokenization
        formatted_batch = [
            tokenizer.apply_chat_template([{"role": "user", "content": t}], tokenize=False, add_generation_prompt=True)
            for t in batch
        ]
        
        enc = tokenizer(
            formatted_batch, return_tensors="pt", padding=True, truncation=True, max_length=128
        ).to(model.device)
        
        # Forward pass triggers hooks
        model(**enc)
        
        # Stack layers: [Batch, Layers, Heads, Dim]
        batch_acts = np.stack(layer_outputs, axis=1)
        activations.append(batch_acts)

    for h in hooks: h.remove()
    return np.concatenate(activations, axis=0)

# ==========================================
# 3. PARALLEL METRICS ENGINE (CPU)
# ==========================================
def calculate_metrics_for_single_head(head_data, party_labels):
    """
    Calculates all 5 metrics for a SINGLE head.
    This function will be mapped across 1024 heads in parallel.
    """
    # Filter Valid Data
    valid_mask = np.isin(party_labels, [100, 200])
    X = head_data[valid_mask]
    y = party_labels[valid_mask]
    
    # Centering for PCA/Covariance
    X_centered = X - np.mean(X, axis=0)
    
    results = {}
    
    # --- BLOCK A: PCA BASED METRICS (3 in 1) ---
    # We run PCA once to get eigenvalues, used for Dispersion, PC1, and Intrinsic Dim
    try:
        pca = PCA(n_components=10) # We only need top eigenvalues
        pca.fit(X_centered)
        evals = pca.explained_variance_
        
        # 3. Total Dispersion (Sum of variance/eigenvalues)
        results['Total_Dispersion'] = np.sum(evals)
        
        # 4. Explained Variance Ratio of PC1
        results['PC1_Ratio'] = pca.explained_variance_ratio_[0]
        
        # 5. Intrinsic Dimensionality (Participation Ratio)
        sum_evals = np.sum(evals)
        sum_sq_evals = np.sum(evals**2)
        if sum_sq_evals > 0:
            results['Intrinsic_Dim'] = (sum_evals**2) / sum_sq_evals
        else:
            results['Intrinsic_Dim'] = 0.0
            
    except Exception:
        results['Total_Dispersion'] = 0.0
        results['PC1_Ratio'] = 0.0
        results['Intrinsic_Dim'] = 0.0

    # --- BLOCK B: CLUSTER METRICS ---
    
    # 2. Davies-Bouldin Index
    # (Lower is better separation, so higher polarization means LOWER score usually)
    try:
        if len(np.unique(y)) > 1:
            results['Davies_Bouldin'] = davies_bouldin_score(X, y)
        else:
            results['Davies_Bouldin'] = 10.0 # Bad score
    except:
        results['Davies_Bouldin'] = 10.0

    # 1. Mahalanobis Distance
    try:
        dems = X[y == 100]
        reps = X[y == 200]
        
        if len(dems) > 5 and len(reps) > 5:
            # Pooled Covariance with regularization
            cov_pool = (np.cov(dems, rowvar=False) + np.cov(reps, rowvar=False)) / 2
            cov_pool += np.eye(cov_pool.shape[0]) * 1e-6 # Regularize
            
            inv_cov = inv(cov_pool)
            mu_d, mu_r = np.mean(dems, axis=0), np.mean(reps, axis=0)
            results['Mahalanobis'] = mahalanobis(mu_d, mu_r, inv_cov)
        else:
            results['Mahalanobis'] = 0.0
    except (LinAlgError, ValueError):
        results['Mahalanobis'] = 0.0
        
    return results

# ==========================================
# 4. MAIN EXECUTION LOOP
# ==========================================

# Setup
model, tokenizer = load_model(MODEL_PATH)

df_nom = pd.read_csv(NOMINATE_CSV)
df_nom = df_nom[df_nom['party_code'].isin([100, 200])].dropna(subset=['bioname'])
party_labels = df_nom['party_code'].values

full_results = []

print(f"\nStarting Optimized Pipeline on A100 (Batch Size {BATCH_SIZE})")
print(f"Parallel processing enabled for metrics calculation.")

for topic_name, topic_desc in TOPICS.items():
    t0 = time.time()
    print(f"\n--- Topic: {topic_name} ---")
    
    # 1. Generate Prompts
    prompts = []
    for name in df_nom['bioname']:
        if topic_desc:
            p = f"You are an analyst of U.S. politicans. Generate a statement by {name} regarding {topic_desc}."
        else:
            p = f"You are an analyst of U.S. politicans. Generate a statement by {name}."
        prompts.append(p)
        
    # 2. Extract Heads (GPU Bound)
    # Shape: [N, 32, 32, 128]
    X_heads = extract_heads_batched(model, tokenizer, prompts, batch_size=BATCH_SIZE)
    
    # 3. Parallel Metrics (CPU Bound)
    # Flatten L and H to iterate easily: List of (1024) arrays of shape (N, 128)
    N, L, H, D = X_heads.shape
    flat_heads = [X_heads[:, l, h, :] for l in range(L) for h in range(H)]
    
    print("  > Computing metrics for 1024 heads (Parallel)...")
    
    # Uses all available CPU cores
    metrics_flat = Parallel(n_jobs=-1)(
        delayed(calculate_metrics_for_single_head)(head_data, party_labels) 
        for head_data in flat_heads
    )
    
    # 4. Aggregation & Storage
    # Reshape back to (32, 32) grids for visualization
    metric_grids = {k: np.zeros((L, H)) for k in metrics_flat[0].keys()}
    
    idx = 0
    for l in range(L):
        for h in range(H):
            m = metrics_flat[idx]
            for key in m:
                metric_grids[key][l, h] = m[key]
            idx += 1
            
    # Calculate Averages/Max for Summary
    summary = {"Topic": topic_name}
    for key, grid in metric_grids.items():
        summary[f"Avg_{key}"] = np.mean(grid)
        summary[f"Max_{key}"] = np.max(grid)
        # Store the full grid for heatmaps later
        summary[f"Grid_{key}"] = grid
        
    full_results.append(summary)
    print(f"  > Done in {time.time() - t0:.1f}s. Avg Mahalanobis: {summary['Avg_Mahalanobis']:.4f}")

# ==========================================
# 5. VISUALIZATION
# ==========================================
df_res = pd.DataFrame(full_results)

# Clean up dataframe for display (drop grids)
display_cols = [c for c in df_res.columns if "Grid" not in c]
print("\n=== FINAL SUMMARY ===")
print(df_res[display_cols].to_markdown())

# Example Plot: Mahalanobis vs Intrinsic Dim
fig = px.scatter(
    df_res, 
    x="Avg_Intrinsic_Dim", 
    y="Avg_Mahalanobis", 
    text="Topic",
    size="Avg_Total_Dispersion",
    title="Political Manifold Geometry: Polarization vs Complexity",
    labels={"Avg_Mahalanobis": "Polarization (Mahalanobis)", "Avg_Intrinsic_Dim": "Complexity (Dimensions)"}
)
fig.show()

In [3]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
def adjust_text_positions(x, y, text_list, max_iterations=100, step_size=0.01):
    """
    Iteratively moves text away from:
      1. Its own data point (repulsion)
      2. Other text labels (collision avoidance)
    Returns optimized x_text, y_text arrays.
    """
    n = len(x)
    # Start text positions exactly at data points
    tx = np.array(x, dtype=float)
    ty = np.array(y, dtype=float)
    
    # Normalize coordinates to 0-1 scale for consistent 'force' calculations
    # We will map them back later.
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    # Avoid division by zero
    if x_range == 0: x_range = 1
    if y_range == 0: y_range = 1
    
    tx_norm = (tx - x_min) / x_range
    ty_norm = (ty - y_min) / y_range
    x_norm = (np.array(x) - x_min) / x_range
    y_norm = (np.array(y) - y_min) / y_range

    for _ in range(max_iterations):
        # Calculate forces
        grad_x = np.zeros(n)
        grad_y = np.zeros(n)
        
        for i in range(n):
            # 1. Force pushing text away from its own data point
            # We want it slightly offset, not ON TOP
            dist_self_x = tx_norm[i] - x_norm[i]
            dist_self_y = ty_norm[i] - y_norm[i]
            dist_sq = dist_self_x**2 + dist_self_y**2
            
            # If too close to dot, push away (standard radius)
            target_radius = 0.04 # 4% of plot width
            if dist_sq < target_radius**2:
                 # Push randomly if exactly on top, otherwise radially
                if dist_sq == 0:
                    grad_x[i] += (np.random.random() - 0.5) * 0.1
                    grad_y[i] += (np.random.random() - 0.5) * 0.1
                else:
                    force = (target_radius - np.sqrt(dist_sq)) 
                    grad_x[i] += force * (dist_self_x / np.sqrt(dist_sq))
                    grad_y[i] += force * (dist_self_y / np.sqrt(dist_sq))

            # 2. Force pushing text away from OTHER labels
            for j in range(n):
                if i == j: continue
                
                diff_x = tx_norm[i] - tx_norm[j]
                diff_y = ty_norm[i] - ty_norm[j]
                dist_sq = diff_x**2 + diff_y**2
                
                # Collision radius (approximate text box size)
                min_dist = 0.05 # 5% of plot width
                
                if dist_sq < min_dist**2:
                     if dist_sq == 0:
                        grad_x[i] += (np.random.random() - 0.5) * 0.1
                        grad_y[i] += (np.random.random() - 0.5) * 0.1
                     else:
                        force = (min_dist - np.sqrt(dist_sq)) * 2 # Stronger force for text collision
                        grad_x[i] += force * (diff_x / np.sqrt(dist_sq))
                        grad_y[i] += force * (diff_y / np.sqrt(dist_sq))
                        
        # Apply movements
        tx_norm += grad_x * step_size
        ty_norm += grad_y * step_size
        
        # Clamp to 0-1 to keep inside plot (optional)
        tx_norm = np.clip(tx_norm, 0, 1)
        ty_norm = np.clip(ty_norm, 0, 1)

    # Convert back to original scale
    final_tx = tx_norm * x_range + x_min
    final_ty = ty_norm * y_range + y_min
    
    return final_tx, final_ty

def plot(df_plot):
    # Run the physics engine to get new coordinates for the TEXT ONLY
    # (Assuming adjust_text_positions is defined in your environment)
    new_x, new_y = adjust_text_positions(
        df_plot["Avg_Mahalanobis"].values, 
        df_plot["Avg_Total_Dispersion"].values, 
        df_plot["Topic"].values,
        max_iterations=200, 
        step_size=0.5
    )

    df_plot['text_x'] = new_x
    df_plot['text_y'] = new_y

    # --- 3. PLOTTING (Using Graph Objects for Layering) ---

    fig = go.Figure()

    # Layer 1: The Dots (Markers)
    unique_topics = df_plot['Topic'].unique()
    colors = px.colors.qualitative.Plotly 

    for i, topic in enumerate(unique_topics):
        df_sub = df_plot[df_plot['Topic'] == topic]
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=df_sub['Avg_Mahalanobis'],
            y=df_sub['Avg_Total_Dispersion'],
            mode='markers',
            name=topic,
            # Pass Intrinsic Dim as custom data for the hover
            customdata=df_sub['Avg_Intrinsic_Dim'], 
            marker=dict(
                size=14,  # FIXED SIZE for visualization
                color=color,
                opacity=0.85,
                line=dict(width=1, color='DarkSlateGrey')
            ),
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "Polarization: %{x:.4f}<br>" +
                "Intensity: %{y:.2f}<br>" +
                "Intrinsic Dim: %{customdata:.2f}<br>" + # Added here
                "<extra></extra>" 
            ),
            text=df_sub['Topic']
        ))

    # Layer 2: The Lines (Connecting dots to labels)
    for i, row in df_plot.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['Avg_Mahalanobis'], row['text_x']],
            y=[row['Avg_Total_Dispersion'], row['text_y']],
            mode='lines',
            line=dict(color='grey', width=0.3),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Layer 3: The Text (at new optimized positions)
    fig.add_trace(go.Scatter(
        x=df_plot['text_x'],
        y=df_plot['text_y'],
        mode='text',
        text=df_plot['Topic'],
        textfont=dict(size=11, color='black'),
        showlegend=False,
        hoverinfo='skip'
    ))

    # --- 4. LAYOUT & ANNOTATIONS ---

    fig.update_layout(
        title="<b>What Politicans Say about Everyday Topics</b><br>X: Polarization | Y: Total Disagreement",
        template="plotly_white",
        height=800,
        width=1000,
        xaxis_title="Polarization (Mahalanobis Distance)",
        yaxis_title="Total Discourse Disagreement (Dispersion)",
        showlegend=True
    )

    # Add Quadrants
    mid_x = df_plot['Avg_Mahalanobis'].median()
    mid_y = df_plot['Avg_Total_Dispersion'].median()

    fig.add_hline(y=mid_y, line_dash="dot", line_color="grey", opacity=0.5)
    fig.add_vline(x=mid_x, line_dash="dot", line_color="grey", opacity=0.5)

    # Add Quadrant Labels (Watermarks)
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].max(), y=df_plot['Avg_Total_Dispersion'].max(),
                    text="<b>High Polarization<br>High Variance</b>", showarrow=False, align="right", opacity=0.3,
                    xref="x", yref="y", xanchor="right", yanchor="top")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].min(), y=df_plot['Avg_Total_Dispersion'].max(),
                    text="<b>Low Polarization<br>High Variance</b>", showarrow=False, align="left", opacity=0.3,
                    xref="x", yref="y", xanchor="left", yanchor="top")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].max(), y=df_plot['Avg_Total_Dispersion'].min(),
                    text="<b>High Polarization<br>Low Variance</b>", showarrow=False, align="right", opacity=0.3,
                    xref="x", yref="y", xanchor="right", yanchor="bottom")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].min(), y=df_plot['Avg_Total_Dispersion'].min(),
                    text="<b>Low Polarization<br>Low Variance</b>", showarrow=False, align="left", opacity=0.3,
                    xref="x", yref="y", xanchor="left", yanchor="bottom")

    fig.show()
    fig.write_html("us_cultural_geometry_all.html", include_plotlyjs="cdn")

In [ ]:
anes_path = "policy_polarization.csv"

df_anes = pd.read_csv(anes_path, index_col=0)
df_anes = df_anes.rename(columns={"mahalanobis_distance": "ANES_Mahalanobis"})
df_anes = df_anes.reset_index().rename(columns={"index": "Topic"})
# Keep only Topic + Avg_Mahalanobis from LLM results

df_llm = df_res[["Topic", "Avg_Mahalanobis"]].copy()

# Inner join ensures exact key matching
df_merged = df_llm.merge(
    df_anes,
    left_on="Topic",
    right_on="Topic",
    how="inner"
)

pearson_corr = df_merged["Avg_Mahalanobis"].corr(
    df_merged["ANES_Mahalanobis"],
    method="pearson"
)

print(f"Pearson correlation: {pearson_corr:.3f}")
spearman_corr = df_merged["Avg_Mahalanobis"].corr(
    df_merged["ANES_Mahalanobis"],
    method="spearman"
)

print(f"Spearman correlation: {spearman_corr:.3f}")
df_merged["LLM_rank"] = df_merged["Avg_Mahalanobis"].rank(ascending=False)
df_merged["ANES_rank"] = df_merged["ANES_Mahalanobis"].rank(ascending=False)

print(df_merged.sort_values("Avg_Mahalanobis", ascending=False).to_markdown())


Pearson correlation: 0.162
Spearman correlation: 0.120
|    | Topic         |   Avg_Mahalanobis |   ANES_Mahalanobis |   LLM_rank |   ANES_rank |
|---:|:--------------|------------------:|-------------------:|-----------:|------------:|
| 32 | ft_trans      |           1.90715 |           1.06988  |          1 |          23 |
| 10 | ar_ban        |           1.8919  |           0.940938 |          2 |          31 |
| 39 | gun_imp       |           1.88052 |           0.274799 |          3 |          40 |
| 26 | obamacare     |           1.84401 |           2.40815  |          4 |           1 |
| 30 | clim_imp      |           1.84355 |           1.5654   |          5 |           8 |
|  2 | paid_leave    |           1.84326 |           0.713889 |          6 |          36 |
| 21 | trans_bath    |           1.84265 |           1.27614  |          7 |          15 |
| 38 | ghg_emiss     |           1.83684 |           1.16936  |          8 |          21 |
|  8 | abortion      |           1.